**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Classical Forecasting: AR, MA & ARIMA

The linear baseline that beat the LSTM in the [RNN workshop](./Intro_RNN.ipynb) finally gets its own theory: AR and MA models, the ACF/PACF fingerprints that identify them, differencing for trends, and honest forecast intervals — all implemented from scratch.

## 1. Pre-requisites

- [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) S1 (WSS, autocorrelation).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2 (least squares).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *AR & MA Models and Their Fingerprints* (~40 min)
**Goal:** read ACF/PACF plots to identify a model, then fit it by Yule-Walker/least squares.
**Builds on:** [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb) S1. &nbsp; **Feeds into:** Session 2 (ARIMA & forecasting).

---

## 2. Two Kinds of Memory

💡 **Intuition.** **AR($p$)** — today is a weighted echo of the last $p$ days plus a shock: $x_t = \sum_i \phi_i x_{t-i} + \varepsilon_t$. Memory *recirculates* (IIR!), so the autocorrelation decays gradually, forever. **MA($q$)** — today is a blend of the last $q$ *shocks*: $x_t = \varepsilon_t + \sum_j \theta_j \varepsilon_{t-j}$. Memory is a conveyor belt (FIR!): the ACF cuts to zero *dead* after lag $q$. The **PACF** (correlation at lag $k$ after regressing out lags $1..k{-}1$) mirrors this: AR($p$) cuts off after $p$, MA tails off. Fingerprint table:

| | ACF | PACF |
|---|---|---|
| AR($p$) | tails off | **cuts off at $p$** |
| MA($q$) | **cuts off at $q$** | tails off |

In [2]:
def acf(x, nlags=25):
    x = x - x.mean()
    r = np.correlate(x, x, "full")[len(x)-1:]
    return r[:nlags+1] / r[0]

def pacf(x, nlags=25):
    out = [1.0]
    for k in range(1, nlags+1):
        X = np.column_stack([x[k-j-1:len(x)-j-1] for j in range(k)])
        phi, *_ = np.linalg.lstsq(X, x[k:], rcond=None)
        out.append(phi[-1])
    return np.array(out)

N = 3000
ar2 = sig.lfilter([1], [1, -1.1, 0.5], rng.standard_normal(N))         # AR(2)
ma2 = sig.lfilter([1, 0.8, 0.6], [1], rng.standard_normal(N))          # MA(2)

fig, axes = plt.subplots(2, 2, figsize=(9.5, 4.2))
ci = 1.96/np.sqrt(N)
for row, (name, x_) in zip(axes, [("AR(2)", ar2), ("MA(2)", ma2)]):
    for ax, (fn, ttl) in zip(row, [(acf, "ACF"), (pacf, "PACF")]):
        vals = fn(x_)
        ax.stem(vals, basefmt=" ")
        ax.axhspan(-ci, ci, alpha=0.15, color="gray")
        ax.set_title(f"{name}: {ttl}")
plt.tight_layout(); plt.show()
print("read the fingerprints: AR(2) → PACF cuts at 2;  MA(2) → ACF cuts at 2")

read the fingerprints: AR(2) → PACF cuts at 2;  MA(2) → ACF cuts at 2


/tmp/ipykernel_2047681/1460871271.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [3]:
# Fit the AR(2) two ways and recover the coefficients [1.1, −0.5]
# (a) Yule-Walker: autocorrelation + Toeplitz solve
from scipy.linalg import solve_toeplitz
r = acf(ar2, 3)
phi_yw = solve_toeplitz(r[:2], r[1:3])
# (b) conditional least squares: regress x_t on (x_{t-1}, x_{t-2})
X = np.column_stack([ar2[1:-1], ar2[:-2]])
phi_ls, *_ = np.linalg.lstsq(X, ar2[2:], rcond=None)
print(f"Yule-Walker  φ = {phi_yw.round(3)}")
print(f"least squares φ = {phi_ls.round(3)}   (truth [1.1, -0.5])")

Yule-Walker  φ = [ 1.099 -0.497]
least squares φ = [ 1.101 -0.497]   (truth [1.1, -0.5])


---
### 🕐 Session 2 of 2 — *ARIMA: Trends, Fitting & Honest Forecasts* (~40 min)
**Goal:** difference away nonstationarity; forecast with widening uncertainty cones.
**Builds on:** Session 1.

---

## 3. The I in ARIMA

💡 **Intuition.** AR/MA theory assumes [stationarity](../Intro_DSP/Statistical_Signal_Processing.ipynb) — but real series trend and wander. **Differencing** ($\nabla x_t = x_t - x_{t-1}$) turns a random-walk-with-drift into a stationary series; do it $d$ times and you have ARIMA($p,d,q$): difference, model the stationary residue, un-difference the forecasts. And know thy landmark: ARIMA(0,1,0) is the random walk, whose best forecast is *today's value* — the 'persistence' baseline from the [RNN bake-off](./Intro_RNN.ipynb).

In [4]:
# A trending, wandering series: drift + random walk + AR(2) wiggle
Nw = 700
walk = np.cumsum(0.08 + 0.5*rng.standard_normal(Nw))
wiggle = sig.lfilter([1], [1, -0.6, 0.3], rng.standard_normal(Nw))
y = walk + wiggle

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.6))
axes[0].plot(y); axes[0].set_title("raw: trending, nonstationary")
axes[1].plot(np.diff(y)); axes[1].set_title("after one difference: stationary, modelable")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2047681/3658402221.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [5]:
# ARIMA(2,1,0) by hand: difference → fit AR(2) + mean → forecast → integrate back
train, test = y[:600], y[600:]
dy = np.diff(train)
mu = dy.mean()
X = np.column_stack([dy[1:-1] - mu, dy[:-2] - mu])
phi, *_ = np.linalg.lstsq(X, dy[2:] - mu, rcond=None)
resid = (dy[2:] - mu) - X @ phi
s2 = resid.var()

# iterate the forecast H steps ahead, tracking variance growth
H = len(test)
d_hist = list(dy[-2:] - mu)
fc_d, var_d = [], []
psi = [1.0]                                              # MA(∞) weights of the AR fit
for h_i in range(H):
    fc_d.append(phi[0]*d_hist[-1] + phi[1]*d_hist[-2])
    d_hist.append(fc_d[-1])
    if h_i > 0:
        psi.append(phi[0]*psi[-1] + (phi[1]*psi[-2] if len(psi) > 1 else 0))
    var_d.append(s2 * np.sum(np.square(psi)))
fc_d = np.array(fc_d) + mu
fc = train[-1] + np.cumsum(fc_d)                          # integrate the differences back
var_path = np.cumsum(var_d)                               # variances of a sum of forecast errors (approx: independent)
band = 1.96*np.sqrt(var_path)

plt.figure(figsize=(9, 3))
plt.plot(np.arange(500, 600), train[500:], "k", linewidth=1, label="history")
plt.plot(np.arange(600, 700), test, "k--", linewidth=1, label="future (truth)")
plt.plot(np.arange(600, 700), fc, "C1", label="ARIMA(2,1,0) forecast")
plt.fill_between(np.arange(600, 700), fc-band, fc+band, alpha=0.2, color="C1", label="95% cone")
plt.legend(); plt.title("The honest signature of a good forecast: a cone that widens like √h")
plt.tight_layout(); plt.show()

inside = np.mean(np.abs(test - fc) <= band)
print(f"fraction of future inside the 95% cone: {inside:.0%}")

fraction of future inside the 95% cone: 100%


/tmp/ipykernel_2047681/4091437595.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The widening cone is the *point*: a forecaster that doesn't confess growing uncertainty is lying. (The exact interval recursion uses the ψ-weights above; our independence approximation is slightly conservative — a good exercise is deriving the exact one.)

**Model checking:** after fitting, the residuals should be white — run their ACF and a [periodogram](../Intro_DSP/Statistical_Signal_Processing.ipynb); structure left in residuals = model too small.

## 4. Conclusion

ACF/PACF fingerprints identify the model; Yule-Walker/least squares fit it; differencing tames trends; ψ-weights price the uncertainty. This is the baseline that every fancy forecaster must beat — and, as the [RNN workshop](./Intro_RNN.ipynb) showed, often doesn't.

---
## Where next

- [Recurrent Neural Networks](./Intro_RNN.ipynb) — the nonlinear challenger, now with its baseline fully understood.
- [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) — AR spectra: these models as PSD estimators.